In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cse250a.modules.config import config
from cse250a.modules.core import TIME_CARDS, Gas, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.models import DiscreteBN, HourOnlyBaseline

In [ ]:
df = load_air_quality( config.paths.data_dir / "air_quality" / "AirQualityUCI.csv")
print(df.shape)
df.isna().mean().round(3).rename("fraction missing").to_frame().T

In [ ]:
target = Gas.CO
weather = (Weather.T, Weather.RH)
sensors = tuple(Sensor)

before = len(df)
df = df.dropna(subset=[Time.HOUR, *weather])
print(f"dropped {before - len(df)} rows with missing weather; {len(df)} left")

In [ ]:
train_raw, val_raw, test_raw = blocked_split(df, seed=config.seed)
print({name: len(d) for name, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]})

In [ ]:
binned = [target, *sensors, *weather]
disc = Discretizer(n_bins=7).fit(train_raw, binned)
train, val, test = (disc.transform(d) for d in (train_raw, val_raw, test_raw))

cards = {**disc.cards, **TIME_CARDS}
print("cards:", {str(k): v for k, v in cards.items() if k in binned or k == Time.HOUR})
train[binned].head()

In [ ]:
network = Network(
    structure={
        target: (Time.HOUR,),
        **{s: (target, *weather) for s in sensors},
    },
    cards=cards,
).validate(train.columns)

# print("inputs:", network.inputs)
# print("\n")
print(network)

In [ ]:
bn = DiscreteBN(network).fit(train)
r = bn.result
print(f"{r.n_iter} iterations, converged: {r.converged}, monotone: {r.is_monotone()}")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(r.history, marker="o")
ax.set_xlabel("EM iteration"); ax.set_ylabel("log-lik + log-prior")
plt.show()

In [ ]:
# P(CO | hour), first hours
bn.cpts[target].to_frame().round(2).head(6) 

In [ ]:
# P(S1 | CO, T, RH)
bn.cpts[Sensor.S1].to_frame().round(2).head(9)

In [ ]:
evidence = (Time.HOUR, *weather, *sensors)

proba = bn.predict_proba(val, target, evidence)
compare = proba.round(2).assign(predicted=proba.idxmax(axis=1), true=val[target])
compare.head(8)

In [ ]:
try:
    bn.predict_proba(val, target, (*evidence, Gas.CO))
except ValueError as e:
    print("ValueError:", e)

In [ ]:
hour = HourOnlyBaseline(cards=cards).fit(train)
metrics = ["n", "accuracy", "macro_f1", "bere", "mean_log_likelihood"]

results = pd.DataFrame(
    {m.name: {k: m.evaluate(val, target, evidence)[k] for k in metrics} for m in (bn, hour)}
).T
results.round(3)

In [ ]:
pd.DataFrame(bn.evaluate(val, target, evidence)["confusion"]).rename_axis(index="true", columns="pred")

In [ ]:
rows = {"all sensors": bn.evaluate(val, target, evidence)}
for s in sensors:
    rows[f"without {s}"] = bn.evaluate(val, target, tuple(e for e in evidence if e != s))
pd.DataFrame(rows).T[["accuracy", "mean_log_likelihood"]].astype(float).round(3)

## 10. From bins back to a concentration

The model predicts **bins**, but the goal is the **concentration** of CO in mg/m³. The posterior gives a probability for each bin, and each bin stands for a typical concentration, so the natural estimate is the **probability-weighted average** of those typical values:

$$\widehat{\text{CO}}_t \;=\; \sum_{k} P(\text{bin}_k \mid \text{evidence}_t)\;\cdot\;\bar{c}_k$$

where $\bar{c}_k$ is the mean raw concentration of the **training** hours that fell in bin $k$. For example, if an hour's posterior is (0.1, 0.3, 0.6) and the bin means are (0.8, 1.9, 3.7) mg/m³, the estimate is 0.1·0.8 + 0.3·1.9 + 0.6·3.7 = 2.87 mg/m³. A confident posterior lands near one bin's mean; an unsure one lands in between, which is closer on average when the model can't tell.

This turns the BN into a concentration estimator we can score in real units (MAE, RMSE), against the raw reference readings.

**Step 1: the bin means $\bar{c}_k$, from train only.** `train` (binned) and `train_raw` share an index, so grouping the raw values by their bin gives each bin's mean. Hours with missing CO have a NaN group and are dropped.

In [ ]:
evidence_s1 = (Sensor.S1, Time.HOUR, *weather)      # weather = (Weather.T, Weather.RH)

proba = bn.predict_proba(val, Gas.CO, evidence_s1)  # P(CO | S1, hour, T, RH): one column per bin
bn.evaluate(val, Gas.CO, evidence_s1)               # scored the same way as before

**Step 2: the estimate per hour.** Each row of `proba` is (P(bin 0), P(bin 1), P(bin 2)); a matrix-vector product does Σₖ P(binₖ) · c̄ₖ for every row at once. `c_bar.loc[proba.columns]` lines the bin means up with the probability columns.

In [ ]:
c_bar = train_raw[Gas.CO].groupby(train[Gas.CO]).mean()   # index: bin 0, 1, 2 → mean mg/m³
c_bar

In [ ]:
proba = bn.predict_proba(val, Gas.CO, evidence)
co_hat = pd.Series(
    proba.to_numpy() @ c_bar.loc[proba.columns].to_numpy(),
    index=val.index, name="co_hat"
)

In [ ]:
truth = val_raw[Gas.CO]
seen = truth.notna()

err = co_hat[seen] - truth[seen]
print(f"MAE  {err.abs().mean():.3f} mg/m³")
print(f"RMSE {np.sqrt((err ** 2).mean()):.3f} mg/m³")

In [ ]:
def concentration_error(model, evidence):
    proba = model.predict_proba(val, Gas.CO, evidence)
    co_hat = proba.to_numpy() @ c_bar.loc[proba.columns].to_numpy()
    err = co_hat[seen.to_numpy()] - truth[seen].to_numpy()
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean())}

pd.DataFrame({
    "bn, all sensors": concentration_error(bn, evidence),
    "bn, S1 only": concentration_error(bn, evidence_s1),
    "hour baseline": concentration_error(hour, evidence),
}).T.round(3)

In [ ]:
pred = proba.idxmax(axis=1)
off = (pred[seen] - val.loc[seen, Gas.CO]).abs()

print("mean |bin error|:", off.mean().round(3))
for k in (0, 1, 2):
    print(f"within {k} bin(s): {(off <= k).mean():.3f}")